# 02 - EDA univariado gráfico

En esta etapa analizaremos **cada variable individualmente**.

## ¿Qué buscamos?

- Forma de la distribución.
- Tendencia central.
- Dispersión.
- Asimetría.
- Frecuencia de categorías.
- Posibles valores atípicos.

> **Importante:** todavía no eliminaremos ni corregiremos observaciones. El objetivo sigue siendo diagnóstico.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

DATA_FILE = Path("cardio_train.csv")

if not DATA_FILE.exists():
    raise FileNotFoundError("Coloca 'cardio_train.csv' en la misma carpeta del notebook.")

df = pd.read_csv(DATA_FILE, sep=";")
df["age_years"] = df["age"] / 365.25
df["bmi"] = df["weight"] / ((df["height"] / 100) ** 2)

df.head()

## 1. Resumen estadístico de variables numéricas

Incluimos los percentiles 1 y 99 para estudiar qué tan lejos están los extremos respecto a la mayor parte de los datos.

In [ ]:
numeric_cols = ["age_years", "height", "weight", "ap_hi", "ap_lo", "bmi"]

df[numeric_cols].describe(
    percentiles=[0.01, 0.25, 0.50, 0.75, 0.99]
).T.round(2)

## 2. Funciones para visualizar variables numéricas

Para cada variable utilizaremos:

- **Histograma:** muestra la forma de la distribución.
- **Boxplot:** ayuda a detectar valores alejados del comportamiento central.

Para algunas variables mostraremos además una vista **P1-P99**. Esto **no elimina datos**: solo permite observar mejor el centro de la distribución cuando existen extremos muy grandes.

In [ ]:
def histogram(series, title, xlabel, bins=30, central=False):
    data = series.dropna()

    if central:
        q01 = data.quantile(0.01)
        q99 = data.quantile(0.99)
        data = data[(data >= q01) & (data <= q99)]
        title += " - vista central (P1-P99)"

    plt.figure(figsize=(8, 5))
    plt.hist(data, bins=bins, edgecolor="black")
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel("Frecuencia")
    plt.show()


def boxplot(series, title, xlabel, central=False):
    data = series.dropna()

    if central:
        q01 = data.quantile(0.01)
        q99 = data.quantile(0.99)
        data = data[(data >= q01) & (data <= q99)]
        title += " - vista central (P1-P99)"

    plt.figure(figsize=(8, 3.5))
    plt.boxplot(data, vert=False)
    plt.title(title)
    plt.xlabel(xlabel)
    plt.show()

## 3. Edad

In [ ]:
histogram(df["age_years"], "Distribución de la edad", "Edad (años)")

In [ ]:
boxplot(df["age_years"], "Boxplot de la edad", "Edad (años)")

### ¿Qué observar?

La edad es una variable continua cuyo rango es aproximadamente de 30 a 65 años. Aquí interesa observar si la muestra se concentra más en determinados grupos de edad y si existen irregularidades evidentes.

## 4. Altura

In [ ]:
histogram(df["height"], "Distribución de la altura", "Altura (cm)")

In [ ]:
boxplot(df["height"], "Boxplot de la altura", "Altura (cm)")

In [ ]:
histogram(df["height"], "Distribución de la altura", "Altura (cm)", central=True)

La vista completa permite detectar extremos; la vista P1-P99 permite observar la distribución habitual sin que esos extremos compriman el eje.

## 5. Peso

In [ ]:
histogram(df["weight"], "Distribución del peso", "Peso (kg)")

In [ ]:
boxplot(df["weight"], "Boxplot del peso", "Peso (kg)")

In [ ]:
histogram(df["weight"], "Distribución del peso", "Peso (kg)", central=True)

## 6. Presión arterial sistólica (`ap_hi`)

In [ ]:
histogram(df["ap_hi"], "Distribución de presión sistólica", "Presión sistólica (mmHg)", bins=40)

In [ ]:
boxplot(df["ap_hi"], "Boxplot de presión sistólica", "Presión sistólica (mmHg)")

In [ ]:
histogram(
    df["ap_hi"],
    "Distribución de presión sistólica",
    "Presión sistólica (mmHg)",
    bins=40,
    central=True
)

En esta variable es especialmente importante comparar la vista completa con la central. Los valores extremos detectados en el EDA inicial pueden hacer que la distribución normal sea casi invisible.

## 7. Presión arterial diastólica (`ap_lo`)

In [ ]:
histogram(df["ap_lo"], "Distribución de presión diastólica", "Presión diastólica (mmHg)", bins=40)

In [ ]:
boxplot(df["ap_lo"], "Boxplot de presión diastólica", "Presión diastólica (mmHg)")

In [ ]:
histogram(
    df["ap_lo"],
    "Distribución de presión diastólica",
    "Presión diastólica (mmHg)",
    bins=40,
    central=True
)

## 8. IMC (`bmi`)

In [ ]:
histogram(df["bmi"], "Distribución del IMC", "IMC (kg/m²)", bins=40)

In [ ]:
boxplot(df["bmi"], "Boxplot del IMC", "IMC (kg/m²)")

In [ ]:
histogram(df["bmi"], "Distribución del IMC", "IMC (kg/m²)", bins=40, central=True)

Recuerda que el IMC se deriva de altura y peso. Si alguna de esas variables contiene errores, el IMC heredará y amplificará esos problemas.

## 9. Variables categóricas y binarias

Para estas variables utilizaremos gráficos de barras, porque interesa comparar la frecuencia de cada categoría.

In [ ]:
categorical_cols = ["gender", "cholesterol", "gluc", "smoke", "alco", "active", "cardio"]

for col in categorical_cols:
    counts = df[col].value_counts().sort_index()

    print(f"\n{col}")
    display(
        pd.DataFrame({
            "frecuencia": counts,
            "porcentaje": (counts / len(df) * 100).round(2)
        })
    )

    plt.figure(figsize=(7, 4.5))
    plt.bar(counts.index.astype(str), counts.values)
    plt.title(f"Distribución de {col}")
    plt.xlabel(col)
    plt.ylabel("Frecuencia")
    plt.show()

## 10. Asimetría

La asimetría (*skewness*) ayuda a cuantificar algo que ya podemos observar visualmente.

- Cerca de **0** → distribución aproximadamente simétrica.
- **Positiva** → cola hacia valores altos.
- **Negativa** → cola hacia valores bajos.

Valores extremadamente altos pueden estar provocados por outliers.

In [ ]:
df[numeric_cols].skew().sort_values(ascending=False).round(3)

## 11. Cuantiles seleccionados

In [ ]:
df[numeric_cols].quantile(
    [0.00, 0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99, 1.00]
).T.round(2)

## 12. ¿Qué debemos concluir del EDA univariado?

Al terminar esta etapa deberíamos poder responder:

1. ¿Cuál es la distribución habitual de cada variable?
2. ¿Existen asimetrías?
3. ¿Existen valores extremos?
4. ¿Qué extremos parecen plausibles y cuáles requieren investigación?
5. ¿Las variables categóricas están muy concentradas en alguna categoría?
6. ¿La variable objetivo está balanceada?

### Regla metodológica importante

Un boxplot que marca un dato como *outlier* **no significa que debamos eliminarlo**.

El criterio IQR es estadístico. En variables clínicas, la decisión debe considerar además:

- plausibilidad fisiológica,
- posibles errores de captura,
- definición del dataset,
- literatura o criterios clínicos,
- impacto sobre el modelo.

El próximo paso será el **EDA bivariado**, donde estudiaremos cómo cambia cada característica según `cardio = 0` o `cardio = 1`.